In [ ]:
#Upload MatData
from google.colab import files
uploaded = files.upload()  

Saving MatData.txt to MatData.txt


In [2]:
slicing_code = r'''
#include <stdio.h>
#include <stdlib.h>

int main(int argc, char **argv) {
    if (argc != 3) {
        printf("Usage: %s <compCount> <threadCount>\n", argv[0]);
        return 1;
    }

    long long compCount = atoll(argv[1]);
    int threadCount = atoi(argv[2]);

    int *sliceList = (int*)malloc(sizeof(int) * threadCount);
    int remainder = (int)(compCount % threadCount);

    for (int i = 0; i < threadCount; i++) {
        sliceList[i] = (int)(compCount / threadCount);
    }
    for (int j = 0; j < remainder; j++) {
        sliceList[j] = sliceList[j] + 1;
    }

    int *startList = (int*)malloc(sizeof(int) * threadCount);
    int *endList   = (int*)malloc(sizeof(int) * threadCount);

    for (int k = 0; k < threadCount; k++) {
        if (k == 0) {
            startList[k] = 0;
            endList[k]   = startList[k] + sliceList[k] - 1;
        } else {
            startList[k] = endList[k-1] + 1;
            endList[k]   = startList[k] + sliceList[k] - 1;
        }
    }

    for (int g = 0; g < threadCount; g++) {
        printf("start = %d  end = %d\n", startList[g], endList[g]);
    }

    free(sliceList);
    free(startList);
    free(endList);
    return 0;
}
'''
open('Slicing.c', 'w').write(slicing_code)
print('Wrote Slicing.c')

Wrote Slicing.c


In [3]:
omp_code = r'''
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <omp.h>
#include <math.h>

/*
Input format per matrix:
rows,cols
v11,v12,...,v1c
v21,v22,...,v2c
...
vr1,vr2,...,vrc

Pairs of matrices A then B. If A.cols != B.rows => print error, skip to next pair.
Results are written to results.txt, each result as:
rows,cols
<row1 comma-separated>
...
<rowN comma-separated>
*/

typedef struct {
    double **data;
    int rows;
    int cols;
} Matrix;

static double** allocate_matrix_data(int rows, int cols) {
    double **m = (double**)malloc(rows * sizeof(double*));
    if (!m) {
        printf("Memory allocation failed\n");
        exit(1);
    }
    for (int i = 0; i < rows; i++) {
        m[i] = (double*)malloc(cols * sizeof(double));
        if (!m[i]) {
            printf("Memory allocation failed for row %d\n", i);
            exit(1);
        }
    }
    return m;
}

static void free_matrix(Matrix *m) {
    if (!m || !m->data) return;
    for (int i = 0; i < m->rows; i++) {
        free(m->data[i]);
    }
    free(m->data);
    m->data = NULL;
    m->rows = 0;
    m->cols = 0;
}

// Read "rows,cols" via fscanf, then read rows*cols doubles separated by commas/newlines.
static int read_matrix(FILE *f, Matrix *out) {
    int r, c;

    // Attempt to read dimensions; return 0 on EOF cleanly
    // Accept optional whitespace around comma.
    int matched = fscanf(f, " %d , %d", &r, &c);
    if (matched == EOF || matched == 0) {
        return 0; // EOF or no more data
    }
    if (matched != 2) {
        // Try to consume the rest of line to avoid infinite loop
        int ch;
        while ((ch = fgetc(f)) != '\n' && ch != EOF) {}
        printf("Error: Could not read matrix dimensions line\n");
        return 0;
    }
    if (r <= 0 || c <= 0) {
        printf("Error: Invalid dimensions: %d x %d\n", r, c);
        return 0;
    }

    out->rows = r;
    out->cols = c;
    out->data = allocate_matrix_data(r, c);

    // Read matrix values row by row; values separated by commas
    for (int i = 0; i < r; i++) {
        for (int j = 0; j < c; j++) {
            // Read number possibly followed by comma
            if (j < c - 1) {
                if (fscanf(f, " %lf ,", &out->data[i][j]) != 1) {
                    printf("Error: Incomplete row %d\n", i);
                    free_matrix(out);
                    return 0;
                }
            } else {
                if (fscanf(f, " %lf", &out->data[i][j]) != 1) {
                    printf("Error: Incomplete row %d\n", i);
                    free_matrix(out);
                    return 0;
                }
            }
        }
        // consume end-of-line if present
        int ch;
        do {
            ch = fgetc(f);
        } while (ch == '\r'); // skip CR from Windows endings
        if (ch != '\n' && ch != EOF) {
            ungetc(ch, f);
        }
    }

    return 1;
}

static void write_matrix(FILE *f, const Matrix *m) {
    fprintf(f, "%d,%d\n", m->rows, m->cols);
    for (int i = 0; i < m->rows; i++) {
        for (int j = 0; j < m->cols; j++) {
            fprintf(f, "%.6f", m->data[i][j]);
            if (j < m->cols - 1) fprintf(f, ",");
        }
        fprintf(f, "\n");
    }
}

int main(int argc, char *argv[]) {
    if (argc != 3) {
        printf("Usage: %s <input_file> <number_of_threads>\n", argv[0]);
        printf("Example: %s MatData.txt 4\n", argv[0]);
        return 1;
    }

    char *input_filename = argv[1];
    int requested_threads = atoi(argv[2]);
    if (requested_threads <= 0) {
        printf("Error: Number of threads must be positive\n");
        return 1;
    }

    FILE *input = fopen(input_filename, "r");
    if (!input) {
        printf("Error: Cannot open input file '%s'\n", input_filename);
        return 1;
    }
    FILE *output = fopen("results.txt", "w");
    if (!output) {
        printf("Error: Cannot create output file 'results.txt'\n");
        fclose(input);
        return 1;
    }

    printf("Matrix Multiplication Program (OpenMP)\n");
    printf("=====================================\n");
    printf("Input file: %s\n", input_filename);
    printf("Requested threads: %d\n\n", requested_threads);

    int pair_index = 0;
    int success_count = 0;

    while (1) {
        Matrix A = {0}, B = {0};
        // Read A
        if (!read_matrix(input, &A)) {
            // Either EOF or error; stop processing
            break;
        }
        // Read B
        if (!read_matrix(input, &B)) {
            printf("Error: Incomplete matrix pair - missing Matrix B\n");
            free_matrix(&A);
            break;
        }

        pair_index++;
        printf("\n--- Processing Matrix Pair %d ---\n", pair_index);
        printf("A: %dx%d, B: %dx%d\n", A.rows, A.cols, B.rows, B.cols);

        if (A.cols != B.rows) {
            printf("Error: Cannot multiply matrices %dx%d and %dx%d\n", A.rows, A.cols, B.rows, B.cols);
            printf("Reason: A.cols (%d) must equal B.rows (%d)\n", A.cols, B.rows);
            free_matrix(&A);
            free_matrix(&B);
            continue;
        }

        Matrix C = {0};
        C.rows = A.rows;
        C.cols = B.cols;
        C.data = allocate_matrix_data(C.rows, C.cols);

        for (int i = 0; i < C.rows; i++) {
            for (int j = 0; j < C.cols; j++) {
                C.data[i][j] = 0.0;
            }
        }

        int max_dimension = (A.rows > B.cols) ? A.rows : B.cols;
        int actual_threads = (requested_threads > max_dimension) ? max_dimension : requested_threads;
        printf("Using %d threads (limited by max dimension %d)\n", actual_threads, max_dimension);

        omp_set_num_threads(actual_threads);

        // Manual contiguous chunking per thread (equal computations) like Slicing.c
        #pragma omp parallel default(none) shared(A,B,C,actual_threads)
        {
            int tid = omp_get_thread_num();
            int nthreads = omp_get_num_threads();

            int base_rows = C.rows / nthreads;
            int remainder = C.rows % nthreads;

            int start_row, end_row, rows_for_this;
            if (tid < remainder) {
                rows_for_this = base_rows + 1;
                start_row = tid * rows_for_this;
            } else {
                rows_for_this = base_rows;
                start_row = remainder * (base_rows + 1) + (tid - remainder) * base_rows;
            }
            end_row = (rows_for_this == 0) ? (start_row - 1) : (start_row + rows_for_this - 1);

            // Multiply rows start_row..end_row
            for (int i = start_row; i <= end_row; i++) {
                for (int j = 0; j < C.cols; j++) {
                    double sum = 0.0;
                    for (int k = 0; k < A.cols; k++) {
                        sum += A.data[i][k] * B.data[k][j];
                    }
                    C.data[i][j] = sum;
                }
            }

            if (rows_for_this > 0) {
                #pragma omp critical
                {
                    printf("Thread %d completed rows %d to %d\n", tid, start_row, end_row);
                }
            }
        }

        write_matrix(output, &C);
        success_count++;

        printf("Matrix pair %d multiplication completed successfully\n", pair_index);

        free_matrix(&A);
        free_matrix(&B);
        free_matrix(&C);
    }

    fclose(input);
    fclose(output);

    printf("\n==========================================\n");
    printf("PROGRAM COMPLETED SUCCESSFULLY\n");
    printf("==========================================\n");
    printf("Total matrix pairs processed: %d\n", pair_index);
    printf("Successful multiplications: %d\n", success_count);
    printf("Results saved to: results.txt\n\n");

    return 0;
}
'''
open('matrix_multiply_omp.c', 'w').write(omp_code)
print('Wrote matrix_multiply_omp.c')

Wrote matrix_multiply_omp.c


In [5]:
!gcc Slicing.c -o slicing_demo
!gcc -O2 -fopenmp matrix_multiply_omp.c -o matrix_multiply_omp -lm

In [8]:
!echo "Running with 4 threads"
!./matrix_multiply_omp MatData.txt 4

!echo
!echo "============================================================"
!echo "Running with 8 threads"
!./matrix_multiply_omp MatData.txt 8

Running with 4 threads
Matrix Multiplication Program (OpenMP)
Input file: MatData.txt
Requested threads: 4


--- Processing Matrix Pair 1 ---
A: 2x4, B: 4x2
Using 2 threads (limited by max dimension 2)
Thread 0 completed rows 0 to 0
Thread 1 completed rows 1 to 1
Matrix pair 1 multiplication completed successfully

--- Processing Matrix Pair 2 ---
A: 9x11, B: 11x9
Using 4 threads (limited by max dimension 9)
Thread 1 completed rows 3 to 4
Thread 3 completed rows 7 to 8
Thread 2 completed rows 5 to 6
Thread 0 completed rows 0 to 2
Matrix pair 2 multiplication completed successfully

--- Processing Matrix Pair 3 ---
A: 13x6, B: 6x13
Using 4 threads (limited by max dimension 13)
Thread 0 completed rows 0 to 3
Thread 1 completed rows 4 to 6
Thread 3 completed rows 10 to 12
Thread 2 completed rows 7 to 9
Matrix pair 3 multiplication completed successfully

--- Processing Matrix Pair 4 ---
A: 10x1, B: 1x10
Using 4 threads (limited by max dimension 10)
Thread 1 completed rows 3 to 5
Thread 2 

In [10]:
!echo "First 50 lines of results:"
!head -50 results.txt

!echo
!echo "Total lines:"
!wc -l results.txt

First 50 lines of results:
2,2
9947047195.529209,14200596809.529045
10519836149.444487,16034405692.067020
9,9
31011369938.519295,35692486797.110977,40653841394.870621,37119512263.544342,42842953929.597488,39459046936.607040,30211185997.392277,46541455110.334343,34304520492.786846
24080248746.934837,30950047968.443741,23444667214.924980,29827064060.174530,20632508745.592812,35857950656.252678,18558586495.492565,31712132456.134750,24437891253.850525
29950962965.743343,32449599240.077686,29768991455.971081,34601572745.962837,27019244485.234554,30997729980.889591,26253744109.200367,33139957790.751572,28919154317.116528
38604584517.468719,31099655203.781036,38702026565.196518,33331277042.610878,30242896894.484764,31965815857.285477,30244593998.380314,41378186062.584160,31494416169.893806
44799877528.651306,38475111809.887062,42816954148.718597,36034108397.048973,36939615796.087875,42481989740.043259,28141513831.482491,45637538469.009041,35726962203.193069
42915661548.738022,34918647181.8868